In [67]:
# ==========================================
# CELL 1: SETUP & CONFIGURATION
# ==========================================
# Computes eps_real,MC(w) and eps_fake,MC(w) -- inclusive (score >= w)
# efficiencies for both LP/MP/HP -- in one pass over the MC. These are
# the two ingredients the multi-WP combination formula needs, applied
# per candidate at analysis time based on its GEN-match status:
#   wSF = prod(tagged at T) SF_T*eps_T/eps_T
#       * prod(tagged at L not T) (SF_L*eps_L - SF_T*eps_T)/(eps_L - eps_T)
#       * prod(not tagged at L) (1 - SF_L*eps_L)/(1 - eps_L)
# NOTE: this measures efficiency in the fake-SF control region
# (n_b_outZH == 1), not the actual analysis signal region. Whether a
# CR-measured eps_real is an acceptable proxy for the SR is still an
# open question -- flagged, not resolved, by this script.
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

hep.style.use(hep.style.CMS)

COFFEA_DIR = "Nominal2017"          # <-- UPDATE THIS PATH
TARGET_ERA = "2017"
LUMI = 41.4  # fb^-1

# Same era-safety pattern as the other scripts: only 2016_PreVFP is
# confirmed. Fill in the other three (or copy 2016_PreVFP's values in, if
# the thresholds turn out to be fixed across years) before running.
THRESHOLDS_BY_ERA = {
    "2016_PreVFP": {"LP": 0.9088, "MP": 0.9737, "HP": 0.9883},
    "2016_PostVFP":    {"LP": 0.9137, "MP": 0.9735, "HP": 0.9883},
    "2017":    {"LP": 0.9105, "MP": 0.9714, "HP": 0.9870},
    "2018":    {"LP": 0.9172, "MP": 0.9734, "HP": 0.9880},
}
if THRESHOLDS_BY_ERA.get(TARGET_ERA) is None:
    raise ValueError(
        f"No WP thresholds defined for era '{TARGET_ERA}' in THRESHOLDS_BY_ERA. "
        f"Fill in the actual LP/MP/HP score cuts for this era before running."
    )
THRESHOLDS = THRESHOLDS_BY_ERA[TARGET_ERA]

PT_BINS  = np.array([200, 300, 400, 500, 600, 800, 1200])
MSD_BINS = np.array([50, 75, 100, 125, 150, 200])

# QCD excluded (not part of this analysis). Signal included and kept
# per-process throughout, so a future signal-rate systematic can
# rescale ttZ/ttH specifically, the same way it does in the fake-SF
# derivation.
bkg_processes = ['VJets', 'tt_B', 'TTBar', 'SingleTop', 'TTX']
sig_processes = ['ttZ', 'ttH']
all_mc_processes = bkg_processes + sig_processes

# No Data needed here at all -- eps_real,MC and eps_fake,MC are purely
# MC-internal quantities; the Data/MC comparison happens in the fake-SF
# derivation, and the real SF is already calibrated externally by BTV.
vars_to_extract = list(set([
    'ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'xbb_genmatch',
    'genWeight', 'norm_weight', 'puWeight', 'btag_sf',
    'ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_reco_sf',
    'topptWeight', 'l1prefire_sf', 'mu_trig_sf', 'ele_trig_sf'
]))
ESSENTIAL_VARS = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets',
                   'xbb_genmatch', 'norm_weight']

# Which region each population's efficiency is measured in. Both are
# SR here: real/fake is purely a GEN-truth boolean (xbb_genmatch),
# unrelated to which region you happen to be looking in, and GEN truth
# exists everywhere in MC, not just the CR. The formula gets applied to
# actual SR candidates, so both eps_real and eps_fake should be
# measured where they're used, same as any other MC-side quantity. The
# CR's job is entirely separate -- it's where SFfake itself gets
# derived (Data-minus-real-contamination vs MC), and that correction
# layer is what actually makes the SR-measured eps_fake,MC(w)
# Data-consistent, not the choice of measurement region. 'SR' requires
# sr_extra_cut() (below) to be filled in first.
REAL_EFF_REGION = "SR"
FAKE_EFF_REGION = "SR"

# ==========================================
# CELL 2: CUTS & WEIGHTING FUNCTIONS
# ==========================================
def common_baseline_cuts(df_):
    """ Cuts shared by both the CR and SR -- per the methodology text,
    the CR is 'all baseline selection criteria (Section 6) except
    requiring exactly 1 extra b-jet', meaning these are common to both
    regions and only the b-jet requirement differs.

    Includes the Z/H candidate passing at least Loose
    (ZH_bbvLscore >= THRESHOLDS['LP']) -- confirmed to be a genuine
    Section-6 baseline eligibility requirement, not region-specific.
    Consequence: eps(LP) will be ~1.0 identically in every bin, for
    both real and fake, in both CR and SR. This is EXPECTED, not a
    bug -- no candidate can exist in this sample with score < LP at
    all, so the 'not tagged at L' branch of the multi-WP formula is
    structurally unreachable for any real candidate in this analysis
    and should never need to be evaluated downstream. """
    base = (
        (df_['n_ak4jets'] >= 5) &
        (df_['ZH_pt']     >= 200) &
        (df_['ZH_M']      >= 50)  &
        (df_['ZH_M']      <= 200) &
        (df_['ZH_bbvLscore'] >= THRESHOLDS["LP"])
    )
    return base


def cr_extra_cut(df_):
    """ The one thing that distinguishes the fake-SF control region
    from the signal region: exactly 1 extra b-jet, chosen specifically
    to make the CR orthogonal to the SR. """
    return df_['n_b_outZH'] == 1


def sr_extra_cut(df_):
    """ Signal-region baseline selection (Section 6): >=2 extra b-jets.

    Deliberately NO cut on ZH_bbvLscore here -- same reason the CR's
    common_baseline_cuts() has never had one ('no tagger score cut
    applied to ensure full denominator'). eps_L, eps_M, eps_T need to
    be measured UNCONDITIONALLY against the full kinematic-baseline
    population, not one already pre-filtered by score. A score >= LP
    pre-filter makes eps_L trivially 1.0 in every bin, turns eps_M/eps_T
    into P(pass M | already passes L) rather than the unconditional
    P(pass M) the multi-WP formula's derivation assumes, and makes the
    formula's 'not tagged at L' term an unevaluated 0/0 (its
    denominator, 1 - eps_L, vanishes identically). If the real Section-6
    SR genuinely hard-requires score >= LP somewhere in the analysis
    chain, that should be a later selection applied AFTER weights are
    computed -- not baked into this measurement sample. """
    cut = (df_['n_b_outZH'] >= 2)
    return cut


REGION_EXTRA_CUT = {"CR": cr_extra_cut, "SR": sr_extra_cut}


def getZhbbWeight(df_):
    """ Full MC weight chain, excluding xbb_sf (circular) and btag_sf
    (applied separately below via shape-preserving renormalization).
    Uses 'l1prefire_sf' (the processor's actual field name), drops
    trigger SFs (not computed anywhere in the processor yet). """
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index)
    weight = df_['norm_weight'].copy()
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_reco_sf',
           'puWeight', 'topptWeight', 'l1prefire_sf', 'mu_trig_sf', 'ele_trig_sf']
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
    return weight

# ==========================================
# CELL 3: DATA EXTRACTION (MC ONLY)
# ==========================================
def load_mc_data(coffea_dir=COFFEA_DIR, year=TARGET_ERA):
    tracked_data = {proc: [] for proc in all_mc_processes}
    all_files = glob.glob(os.path.join(coffea_dir, "*/*.coffea"))
    valid_files = [f for f in all_files if 'nom' in f.lower()]
    print(f"Extracting from {len(valid_files)} nominal files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        subdir = os.path.basename(os.path.dirname(file_path))
        is_data_file = (subdir == 'Data')

        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc: mapped_proc = 'TTBar' if 'tt+B' not in raw_proc else None
            elif 'TTbb' in raw_proc: mapped_proc = 'tt_B' if 'tt+B' in raw_proc else None
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower() or 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'ttX' in raw_proc: mapped_proc = 'TTX'

            if mapped_proc not in all_mc_processes: continue

            # Only take data from Data/, and only take MC from non-Data dirs.
            if is_data_file != (mapped_proc == 'data_obs'):
                print(f"Skipping '{raw_proc}' in {file_path}: "
                      f"{'MC key in Data file' if is_data_file else 'data key in MC file'}")
                continue

            for dataset in filein['columns'][raw_proc].keys():
                is_apv = any(x in dataset for x in ['APV', 'PreVFP', 'HIPM'])
                if year == "2016_PreVFP" and not is_apv: continue
                if year == "2016" and is_apv: continue
                if year in ["2017", "2018"] and year not in dataset: continue

                genweight = genweight_dict.get(dataset, 1.0)
                if isinstance(genweight, dict):
                    genweight = genweight.get(dataset, 1.0)

                try:
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue

                tmp_data = {}
                skip_dataset = False
                for var in vars_to_extract:
                    dict_key = f'events_{var}'
                    if dict_key in nom_dict:
                        tmp_data[var] = np.array(nom_dict[dict_key].value)
                    elif var in ESSENTIAL_VARS:
                        print(f"\U0001F6A8 Missing essential '{dict_key}' in {mapped_proc} ({dataset})")
                        skip_dataset = True
                        break

                if skip_dataset or not tmp_data: continue

                tmp_data['norm_weight'] = tmp_data['norm_weight'] / genweight
                tracked_data[mapped_proc].append(pd.DataFrame(tmp_data))

    data_dict = {}
    for proc in all_mc_processes:
        if not tracked_data[proc]:
            data_dict[proc] = None
            continue
        df = pd.concat(tracked_data[proc], ignore_index=True)

        base_weight = getZhbbWeight(df)
        raw_btag_sf = df.get('btag_sf', pd.Series(1.0, index=df.index)).fillna(1.0)
        sum_before = np.sum(base_weight)
        sum_after = np.sum(base_weight * raw_btag_sf)
        r_ratio = sum_before / sum_after if sum_after > 0 else 1.0
        df['tot_weight'] = base_weight * raw_btag_sf * r_ratio

        data_dict[proc] = df[common_baseline_cuts(df)].copy()

    return data_dict

# ==========================================
# CELL 4: SPLIT REAL / FAKE PER PROCESS
# ==========================================
print("\n--- Starting Data Extraction ---")
current_data = load_mc_data()

real_extra_cut_fn = REGION_EXTRA_CUT[REAL_EFF_REGION]
fake_extra_cut_fn = REGION_EXTRA_CUT[FAKE_EFF_REGION]
print(f"Real efficiency measured in: {REAL_EFF_REGION}   |   Fake efficiency measured in: {FAKE_EFF_REGION}")

real_by_proc, fake_by_proc = {}, {}
for proc in all_mc_processes:
    df = current_data[proc]
    if df is None or df.empty:
        continue
    real_candidates = df[real_extra_cut_fn(df) & (df['xbb_genmatch'] == True)]
    fake_candidates = df[fake_extra_cut_fn(df) & (df['xbb_genmatch'] == False)]
    if not real_candidates.empty:
        real_by_proc[proc] = real_candidates
    if not fake_candidates.empty:
        fake_by_proc[proc] = fake_candidates

print(f"Real candidates by process: {[(p, len(d)) for p, d in real_by_proc.items()]}")
print(f"Fake candidates by process: {[(p, len(d)) for p, d in fake_by_proc.items()]}")

# ==========================================
# CELL 5: COMPUTE INCLUSIVE EFFICIENCY MAPS
# ==========================================
def calculate_2d_inclusive_efficiency(df, wp_cut, weight_col='tot_weight'):
    """ eps(w) = (weighted count with score >= wp_cut) / (weighted
    total), per (pT, mSD) bin, plus sum-of-weights-squared on the
    denominator (for a future effective-N Clopper-Pearson treatment). """
    pt, msd = df['ZH_pt'].values, df['ZH_M'].values
    scores, weights = df['ZH_bbvLscore'].values, df[weight_col].values

    denom, _, _ = np.histogram2d(pt, msd, bins=[PT_BINS, MSD_BINS], weights=weights)
    denom_sumw2, _, _ = np.histogram2d(pt, msd, bins=[PT_BINS, MSD_BINS], weights=weights**2)

    mask = scores >= wp_cut
    num, _, _ = np.histogram2d(pt[mask], msd[mask], bins=[PT_BINS, MSD_BINS], weights=weights[mask])

    eff = np.divide(num, denom, out=np.zeros_like(num), where=(denom > 0))
    return eff, num, denom, denom_sumw2


def plot_eff_map(eff_map, title, ax):
    hep.hist2dplot(eff_map, PT_BINS, MSD_BINS, ax=ax, cmap='viridis', cmin=0, cmax=1)
    ax.set_xlabel(r'$Z/H$ candidate $p_T$ [GeV]')
    ax.set_ylabel(r'$Z/H$ candidate $m_{SD}$ [GeV]')
    ax.set_title(title)
    cbar = plt.colorbar(ax.collections[-1], ax=ax)
    cbar.set_label('Efficiency $\\epsilon$')


def build_population_maps(by_proc, label):
    per_process = {}
    for proc, df in by_proc.items():
        per_process[proc] = {}
        for wp, thresh in THRESHOLDS.items():
            eff, num, denom, denom_sumw2 = calculate_2d_inclusive_efficiency(df, thresh)
            per_process[proc][wp] = {'eff': eff, 'num': num, 'denom': denom, 'denom_sumw2': denom_sumw2}
            mean_eff = np.mean(eff[denom > 0]) if np.any(denom > 0) else 0.0
            print(f"  [{label}] {proc:10} {wp:3} (>= {thresh:.4f}): Mean Eff = {mean_eff:.3f}")

    aggregate = {}
    for wp in THRESHOLDS:
        agg_num, agg_denom, agg_sumw2 = None, None, None
        for proc in per_process:
            entry = per_process[proc][wp]
            agg_num   = entry['num'].copy()          if agg_num   is None else agg_num   + entry['num']
            agg_denom = entry['denom'].copy()         if agg_denom is None else agg_denom + entry['denom']
            agg_sumw2 = entry['denom_sumw2'].copy()   if agg_sumw2 is None else agg_sumw2 + entry['denom_sumw2']
        agg_eff = np.divide(agg_num, agg_denom, out=np.zeros_like(agg_num), where=(agg_denom > 0)) if agg_num is not None else None
        aggregate[wp] = {'eff': agg_eff, 'num': agg_num, 'denom': agg_denom, 'denom_sumw2': agg_sumw2}
        mean_eff = np.mean(agg_eff[agg_denom > 0]) if agg_denom is not None and np.any(agg_denom > 0) else 0.0
        print(f"  [{label}] {'AGGREGATE':10} {wp:3}: Mean Eff = {mean_eff:.3f}")

    return {'per_process': per_process, 'aggregate': aggregate}


print("\n--- Real efficiency (GEN-matched) ---")
print("NOTE: eps(LP) is expected to be ~1.0 in every bin now that the baseline "
      "selection requires ZH_bbvLscore >= LP -- this is not a bug, see "
      "common_baseline_cuts() docstring.")
real_results = build_population_maps(real_by_proc, "real")

print("\n--- Fake efficiency / mistag rate (not GEN-matched) ---")
fake_results = build_population_maps(fake_by_proc, "fake")

# ==========================================
# CELL 6: PLOT & SAVE
# ==========================================
pdf_path = f"EfficiencyMaps_{TARGET_ERA}.pdf"
with PdfPages(pdf_path) as pdf:
    for label, results, title_prefix in [("real", real_results, "Real"), ("fake", fake_results, "Fake")]:
        fig, axes = plt.subplots(1, 3, figsize=(24, 7))
        fig.suptitle(f"{title_prefix} Efficiency \u2014 All Processes Combined", fontsize=20, y=1.05)
        for idx, wp in enumerate(THRESHOLDS):
            plot_eff_map(results['aggregate'][wp]['eff'], f"{wp} (>= {THRESHOLDS[wp]:.4f})", ax=axes[idx])
        hep.cms.label("Preliminary", ax=axes[0], data=False, lumi=LUMI, com=13)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

out_dir = 'efficiency_maps'
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, f'efficiency_maps_{TARGET_ERA}.pkl')
with open(out_file, 'wb') as f:
    pickle.dump({
        'pt_bins': PT_BINS, 'msd_bins': MSD_BINS, 'thresholds': THRESHOLDS,
        'real': real_results, 'fake': fake_results,
    }, f)

print(f"\n--- Complete ---")
print(f"Saved plots to: {pdf_path}")
print(f"Saved numerical matrices to: {out_file}")


--- Starting Data Extraction ---
Extracting from 5 nominal files...
Real efficiency measured in: SR   |   Fake efficiency measured in: SR
Real candidates by process: [('VJets', 927), ('tt_B', 3903), ('TTBar', 253), ('SingleTop', 130), ('TTX', 32100), ('ttZ', 90180), ('ttH', 17753)]
Fake candidates by process: [('VJets', 1520), ('tt_B', 9128), ('TTBar', 24125), ('SingleTop', 575), ('TTX', 146777), ('ttZ', 69332), ('ttH', 23183)]

--- Real efficiency (GEN-matched) ---
NOTE: eps(LP) is expected to be ~1.0 in every bin now that the baseline selection requires ZH_bbvLscore >= LP -- this is not a bug, see common_baseline_cuts() docstring.
  [real] VJets      LP  (>= 0.9105): Mean Eff = 1.000
  [real] VJets      MP  (>= 0.9714): Mean Eff = 0.499
  [real] VJets      HP  (>= 0.9870): Mean Eff = 0.232
  [real] tt_B       LP  (>= 0.9105): Mean Eff = 1.000
  [real] tt_B       MP  (>= 0.9714): Mean Eff = 0.367
  [real] tt_B       HP  (>= 0.9870): Mean Eff = 0.147
  [real] TTBar      LP  (>= 0.9105

In [68]:
# ==========================================
# CELL 1: SETUP & CONFIGURATION
# ==========================================
import os
import glob
import gzip
import pandas as pd
import numpy as np
from scipy.stats import beta
from scipy.stats import chi2 as chi2_dist
import correctionlib
import correctionlib.schemav2 as cs
from coffea.util import load

# --- USER PARAMS ---
COFFEA_DIR = "Nominal2017"          # <-- UPDATE THIS PATH
TARGET_ERA = "2017"
SIGNAL_RATE_VARIATION = 0.20      # +-20%, applied to ttZ + ttH only

# Real-SF file naming has drifted between scripts in this pipeline.
# Try the TARGET_ERA convention first (what the processor's own
# year_map_xbb mapping actually evaluates xbb_era to), fall back to the
# older ALL_DATA-key convention ("2016_PreVFP" etc) with a loud warning
# if that's what's actually on disk.
ERA_TO_LEGACY_REALSF = {"2016_PreVFP": "2016_PreVFP", "2016_PostVFP": "2016_PostVFP", "2017": "2017", "2018": "2018"}
LEGACY_ERA = ERA_TO_LEGACY_REALSF[TARGET_ERA]
REALSF_CANDIDATES = [
    (TARGET_ERA, f"SFBDT_Xbb_{TARGET_ERA}.json", f"SFBDT_Xbb_SF_{TARGET_ERA}"),
    (LEGACY_ERA, f"SFBDT_Xbb_{LEGACY_ERA}.json", f"SFBDT_Xbb_SF_{LEGACY_ERA}"),
]
REAL_SF_FILE, REAL_SF_CORRECTION_NAME = None, None
for _label, _path, _corr_name in REALSF_CANDIDATES:
    if os.path.exists(_path):
        REAL_SF_FILE, REAL_SF_CORRECTION_NAME = _path, _corr_name
        if _label != TARGET_ERA:
            print(f"\u26A0\uFE0F  Found real-SF file under the OLD '{_label}' naming convention "
                  f"('{_path}'). Worth renaming/rebuilding under '{TARGET_ERA}' long-term.")
        break
if REAL_SF_FILE is None:
    print(f"\U0001F6A8 WARNING: no real-SF file found under either naming convention. "
          f"Tried: {[c[1] for c in REALSF_CANDIDATES]}. You must generate the Real SFs first.")

OUTPUT_FAKE_SF_FILE = f"Fake_SF_Xbb_{TARGET_ERA}.json"

# --- Working points: must be bit-for-bit identical to the thresholds
# the real-SF file was actually calibrated with, per era. ---
THRESHOLDS_BY_ERA = {
    "2016_PreVFP": {"LP": 0.9088, "MP": 0.9737, "HP": 0.9883},
    "2016_PostVFP":    {"LP": 0.9137, "MP": 0.9735, "HP": 0.9883},
    "2017":    {"LP": 0.9105, "MP": 0.9714, "HP": 0.9870},
    "2018":    {"LP": 0.9172, "MP": 0.9734, "HP": 0.9880},
}
if THRESHOLDS_BY_ERA.get(TARGET_ERA) is None:
    raise ValueError(
        f"No WP thresholds defined for era '{TARGET_ERA}' in THRESHOLDS_BY_ERA. "
        f"Fill in the actual LP/MP/HP score cuts for this era before running -- "
        f"do not assume they match 2016_PreVFP without checking."
    )
THRESHOLDS = THRESHOLDS_BY_ERA[TARGET_ERA]

# 2D binning in (pT, mSD). PT_BINS deliberately matches the real-SF
# file's own coarse pT edges (200/300/450/10000) -- there's no benefit
# to deriving at finer pT granularity than the calibration being pulled
# in supports, and it also raises per-bin statistics, which was the fix
# for the earlier MP~8 blowup (a near-empty bin dominating a naive mean).
PT_BINS  = np.array([200, 300, 450, 10000])
MSD_BINS = np.array([50, 80, 115, 155, 200])
PT_BIN_CENTERS = 0.5 * (PT_BINS[:-1] + PT_BINS[1:])

# --- Processes ---
# QCD excluded -- not part of this analysis.
# Signal (ttZ, ttH) is included and kept per-process: it genuinely
# contributes both real and fake candidates to this CR, and the +-20%
# "signal rate" systematic has to target ttZ/ttH specifically, not the
# whole real-MC population.
bkg_processes = ['VJets', 'tt_B', 'TTBar', 'SingleTop', 'TTX']
sig_processes = ['ttZ', 'ttH']
mc_processes = bkg_processes + sig_processes
data_process = 'data_obs'
all_processes = mc_processes + [data_process]

# Full weight chain, minus xbb_sf (circular -- this is what's being
# calibrated) and minus btag_sf (applied separately below via
# shape-preserving renormalization; including it here too double-counts it).
vars_to_extract = list(set([
    'ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'xbb_genmatch',
    'genWeight', 'norm_weight', 'puWeight', 'btag_sf',
    'ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_reco_sf',
    'topptWeight', 'l1prefire_sf', 'ele_trig_sf', 'mu_trig_sf'
]))

# Required for every process (MC and Data alike) -- no fallback.
ESSENTIAL_VARS_COMMON = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets']
# Required for MC only -- Data has neither GEN info nor MC weights.
ESSENTIAL_VARS_MC_ONLY = ['xbb_genmatch', 'norm_weight']

if not os.path.exists(COFFEA_DIR):
    print(f"\U0001F6A8 WARNING: COFFEA_DIR '{COFFEA_DIR}' not found!")
else:
    print(f"\u2705 Configuration loaded.")

# ==========================================
# CELL 2: CUTS & EXTRACTION FUNCTIONS
# ==========================================
def cuts(df_):
    """ Fake-SF control region: exactly 1 extra b-jet, plus the shared
    Section-6 baseline requirement that the Z/H candidate passes at
    least Loose (ZH_bbvLscore >= THRESHOLDS['LP']).

    Consequence: the 'Fail' (score < LP) exclusive category is
    structurally empty in this CR -- data_excl['Fail'] and
    fake_excl_nom['Fail'] will both be zero everywhere, which trips the
    existing zero-yield guards below and produces SFfake['Fail'] = NaN
    (written out as an inert 1.0 in the correctionlib file), the same
    way any other zero-yield bin already gets handled. This is
    EXPECTED: if nothing can exist below Loose anywhere in this
    analysis, there's nothing to derive a Fail SF for. """
    base = (
        (df_['n_ak4jets']    >= 5) &
        (df_['n_b_outZH']    == 1) &
        (df_['ZH_pt']        >= 200) &
        (df_['ZH_M']         >= 50)  &
        (df_['ZH_M']         <= 200) &
        (df_['ZH_bbvLscore'] >= THRESHOLDS["LP"])
    )
    return base


def getZhbbWeight(df_):
    """ Full MC weight chain, excluding xbb_sf and btag_sf (see module
    docstring above). Uses 'l1prefire_sf' (the processor's actual field
    name), not 'PrefireWeight'; drops trigger SFs since they aren't
    computed anywhere in the processor yet. """
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index)
    weight = df_['norm_weight'].copy()
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_reco_sf',
           'puWeight', 'topptWeight', 'l1prefire_sf', 'ele_trig_sf', 'mu_trig_sf']
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
    return weight


def load_and_cut_data(coffea_dir=COFFEA_DIR, year=TARGET_ERA):
    tracked_data = {proc: [] for proc in all_processes}
    all_files = glob.glob(os.path.join(coffea_dir, "*/*.coffea"))
    valid_files = [f for f in all_files if 'nom' in f.lower()]
    print(f"Extracting from {len(valid_files)} nominal files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        #subdir = os.path.basename(os.path.dirname(file_path))
        #is_data_file = (subdir == 'Data')

        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc: mapped_proc = 'TTBar' if 'tt+B' not in raw_proc else None
            elif 'TTbb' in raw_proc: mapped_proc = 'tt_B' if 'tt+B' in raw_proc else None
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower() or 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'ttX' in raw_proc: mapped_proc = 'TTX'
            elif 'data' in raw_proc.lower(): mapped_proc = 'data_obs'

            if mapped_proc not in all_processes: continue

            # Only take data from Data/, and only take MC from non-Data dirs.
            #if is_data_file != (mapped_proc == 'data_obs'):
            #    print(f"Skipping '{raw_proc}' in {file_path}: "
            #          f"{'MC key in Data file' if is_data_file else 'data key in MC file'}")
            #    continue

            for dataset in filein['columns'][raw_proc].keys():
                is_apv = any(x in dataset for x in ['APV', 'PreVFP', 'HIPM'])
                if year == "2016_PreVFP" and not is_apv: continue
                if year == "2016_PostVFP" and is_apv: continue
                if year in ["2017", "2018"] and year not in dataset: continue

                genweight = genweight_dict.get(dataset, 1.0)
                if isinstance(genweight, dict):
                    genweight = genweight.get(dataset, 1.0)

                try:
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue

                tmp_data = {}
                skip_dataset = False
                for var in vars_to_extract:
                    dict_key = f'events_{var}'
                    if dict_key in nom_dict:
                        tmp_data[var] = np.array(nom_dict[dict_key].value)
                    else:
                        is_essential = (var in ESSENTIAL_VARS_COMMON) or \
                                       (mapped_proc != 'data_obs' and var in ESSENTIAL_VARS_MC_ONLY)
                        if is_essential:
                            print(f"\U0001F6A8 Missing essential '{dict_key}' in {mapped_proc} ({dataset})")
                            skip_dataset = True
                            break

                if skip_dataset or not tmp_data: continue

                if mapped_proc != 'data_obs' and 'norm_weight' in tmp_data:
                    tmp_data['norm_weight'] = tmp_data['norm_weight'] / genweight

                tracked_data[mapped_proc].append(pd.DataFrame(tmp_data))

    data_dict = {}
    for proc in all_processes:
        if not tracked_data[proc]:
            data_dict[proc] = None
            continue
        df = pd.concat(tracked_data[proc], ignore_index=True)

        if proc != 'data_obs':
            base_weight = getZhbbWeight(df)
            raw_btag_sf = df.get('btag_sf', pd.Series(1.0, index=df.index)).fillna(1.0)
            sum_before = np.sum(base_weight)
            sum_after = np.sum(base_weight * raw_btag_sf)
            r_ratio = sum_before / sum_after if sum_after > 0 else 1.0
            df['tot_weight'] = base_weight * raw_btag_sf * r_ratio
        else:
            df['tot_weight'] = 1.0

        data_dict[proc] = df[cuts(df)].copy()

    return data_dict

# ==========================================
# CELL 3: RUN EXTRACTION & SPLIT REAL/FAKE
# ==========================================
print("\n--- Starting Data Extraction ---")
current_data = load_and_cut_data()

data_df = current_data['data_obs']
if data_df is None or data_df.empty:
    raise RuntimeError("No Data events passing CR cuts -- check Data identification/era filtering.")

mc_real_by_proc, mc_fake_by_proc = {}, {}
for proc in mc_processes:
    df = current_data[proc]
    if df is None or df.empty:
        continue
    if 'xbb_genmatch' not in df.columns:
        print(f"Warning: 'xbb_genmatch' missing for {proc} -- excluding from real/fake split entirely.")
        continue
    mc_real_by_proc[proc] = df[df['xbb_genmatch'] == True]
    mc_fake_by_proc[proc] = df[df['xbb_genmatch'] == False]

mc_real_df = pd.concat(mc_real_by_proc.values(), ignore_index=True) if mc_real_by_proc else pd.DataFrame()
mc_fake_df = pd.concat(mc_fake_by_proc.values(), ignore_index=True) if mc_fake_by_proc else pd.DataFrame()

print(f"Data events passing CR baseline:     {len(data_df):,}")
print(f"MC Real events passing CR baseline:  {len(mc_real_df):,}  (processes: {list(mc_real_by_proc.keys())})")
print(f"MC Fake events passing CR baseline:  {len(mc_fake_df):,}  (processes: {list(mc_fake_by_proc.keys())})")

# ==========================================
# CELL 4: DERIVATION & UNCERTAINTIES
# ==========================================
def get_2d_incl(df, thresh=None, weight_col='tot_weight'):
    """ 2D (pT, mSD) weighted yield and sum-of-weights-squared for
    events with score >= thresh (all events if thresh is None). """
    mask = (df['ZH_bbvLscore'].values >= thresh) if thresh is not None else np.ones(len(df), dtype=bool)
    pt, msd, w = df['ZH_pt'].values[mask], df['ZH_M'].values[mask], df[weight_col].values[mask]
    y, _, _ = np.histogram2d(pt, msd, bins=[PT_BINS, MSD_BINS], weights=w)
    sumw2, _, _ = np.histogram2d(pt, msd, bins=[PT_BINS, MSD_BINS], weights=w**2)
    return y, sumw2


def get_2d_excl(df, wp, weight_col='tot_weight'):
    """ 2D (pT, mSD) weighted yield and sumw2 for one exclusive tagger
    category: Fail / LP-not-MP / MP-not-HP / HP. """
    score = df['ZH_bbvLscore'].values
    if wp == "HP":     mask = score >= THRESHOLDS["HP"]
    elif wp == "MP":   mask = (score >= THRESHOLDS["MP"]) & (score < THRESHOLDS["HP"])
    elif wp == "LP":   mask = (score >= THRESHOLDS["LP"]) & (score < THRESHOLDS["MP"])
    elif wp == "Fail": mask = score < THRESHOLDS["LP"]
    pt, msd, w = df['ZH_pt'].values[mask], df['ZH_M'].values[mask], df[weight_col].values[mask]
    y, _, _ = np.histogram2d(pt, msd, bins=[PT_BINS, MSD_BINS], weights=w)
    sumw2, _, _ = np.histogram2d(pt, msd, bins=[PT_BINS, MSD_BINS], weights=w**2)
    return y, sumw2


def clopper_pearson_interval(k, n, cl=0.6827):
    """ Exact (1-sigma equivalent by default) Clopper-Pearson interval. """
    k, n = np.asarray(k, dtype=float), np.asarray(n, dtype=float)
    alpha = 1 - cl
    with np.errstate(invalid='ignore'):
        lo = np.where(k <= 0, 0.0, beta.ppf(alpha / 2, np.maximum(k, 1e-12), np.maximum(n - k + 1, 1e-12)))
        hi = np.where(k >= n, 1.0, beta.ppf(1 - alpha / 2, k + 1, np.maximum(n - k, 1e-12)))
    return lo, hi


def effective_n(total, total_sumw2):
    """ N_eff = (sum w)^2 / (sum w^2) -- lets weighted MC be treated
    with Clopper-Pearson, which is built for exact binomial counting. """
    return np.divide(total**2, total_sumw2, out=np.zeros_like(total), where=(total_sumw2 > 0))


def stat_band_on_ratio(pass_frac, total, total_sumw2):
    n_eff = effective_n(total, total_sumw2)
    k_eff = n_eff * pass_frac
    return clopper_pearson_interval(k_eff, n_eff)


def per_process_yields(by_proc_dict, extractor, *extractor_args):
    return {proc: extractor(df, *extractor_args) for proc, df in by_proc_dict.items()}


def combine_processes(per_proc_yields, sig_processes, scale=1.0):
    """ Sums {proc: (yield, sumw2)} across processes. Any process in
    sig_processes has its yield scaled by `scale` and its sumw2 scaled
    by `scale**2` (sumw2 scales quadratically under a weight rescaling,
    not linearly -- w' = w*scale implies (w')^2 = w^2 * scale^2). """
    total_y, total_sumw2 = None, None
    for proc, (y, sw2) in per_proc_yields.items():
        factor = scale if proc in sig_processes else 1.0
        y_c, sw2_c = y * factor, sw2 * factor**2
        total_y = y_c.copy() if total_y is None else total_y + y_c
        total_sumw2 = sw2_c.copy() if total_sumw2 is None else total_sumw2 + sw2_c
    return total_y, total_sumw2


def build_exclusive_real_prediction(real_incl, real_sf_syst):
    """ real_incl: {w: (yield, sumw2)} for w in HP/MP/LP/Total (already
    process-combined). real_sf_syst: {wp: 2D array} of SF_real values.
    SF_real is only defined cumulatively (score >= w), so the
    SF-corrected exclusive-category prediction is built as the
    difference of SF-corrected inclusive predictions, not by applying
    an SF directly to an exclusive slice. Each exclusive term is
    clamped at 0 -- mirrors the negative-numerator clamp in the
    standard multi-WP combination formula: a tighter WP's SF exceeding
    a looser WP's SF can otherwise drive an exclusive category
    unphysically negative. """
    scaled_incl = {
        "HP": real_incl["HP"][0] * real_sf_syst["HP"],
        "MP": real_incl["MP"][0] * real_sf_syst["MP"],
        "LP": real_incl["LP"][0] * real_sf_syst["LP"],
        "Total": real_incl["Total"][0],
    }
    scaled_excl = {
        "HP":   scaled_incl["HP"],
        "MP":   scaled_incl["MP"] - scaled_incl["HP"],
        "LP":   scaled_incl["LP"] - scaled_incl["MP"],
        "Fail": scaled_incl["Total"] - scaled_incl["LP"],
    }
    return {c: np.maximum(v, 0.0) for c, v in scaled_excl.items()}


CATEGORIES = ["HP", "MP", "LP", "Fail"]
REAL_THRESH_KEYS = {"HP": THRESHOLDS["HP"], "MP": THRESHOLDS["MP"], "LP": THRESHOLDS["LP"], "Total": None}

# --- Per-process yields (kept per-process so the signal-rate
# systematic can rescale ttZ/ttH specifically, without re-reading data) ---
real_incl_by_proc = {w: per_process_yields(mc_real_by_proc, get_2d_incl, t) for w, t in REAL_THRESH_KEYS.items()}
real_excl_by_proc = {c: per_process_yields(mc_real_by_proc, get_2d_excl, c) for c in CATEGORIES}
fake_excl_by_proc = {c: per_process_yields(mc_fake_by_proc, get_2d_excl, c) for c in CATEGORIES}
fake_total_by_proc = {proc: get_2d_incl(df, None) for proc, df in mc_fake_by_proc.items()}

# --- Nominal (rate_mult = 1.0) aggregates ---
real_incl_nom  = {w: combine_processes(real_incl_by_proc[w], sig_processes, 1.0) for w in REAL_THRESH_KEYS}
real_excl_nom  = {c: combine_processes(real_excl_by_proc[c], sig_processes, 1.0) for c in CATEGORIES}
fake_excl_nom  = {c: combine_processes(fake_excl_by_proc[c], sig_processes, 1.0) for c in CATEGORIES}
fake_total_nom = combine_processes(fake_total_by_proc, sig_processes, 1.0)

data_excl = {c: get_2d_excl(data_df, c) for c in CATEGORIES}
data_total_y, data_total_sumw2 = get_2d_incl(data_df, None)

# --- Real SF: nominal, up, down ---
real_sf = {"nominal": {}, "up": {}, "down": {}}
if REAL_SF_FILE is not None:
    cset = correctionlib.CorrectionSet.from_file(REAL_SF_FILE)
    real_evaluator = cset[REAL_SF_CORRECTION_NAME]
    for syst in ["nominal", "up", "down"]:
        for wp in ["HP", "MP", "LP"]:
            vals = np.array([real_evaluator.evaluate(wp, syst, float(pt)) for pt in PT_BIN_CENTERS])
            real_sf[syst][wp] = vals[:, None]  # broadcast over the mSD axis
else:
    raise RuntimeError("No real-SF file available -- cannot proceed.")

# --- Central value ---
real_excl_predicted_nom = build_exclusive_real_prediction(real_incl_nom, real_sf["nominal"])
sffake_central = {}
for c in CATEGORIES:
    data_fake_pred = np.clip(data_excl[c][0] - real_excl_predicted_nom[c], 0.0, None)
    fake_y = fake_excl_nom[c][0]
    sffake_central[c] = np.divide(data_fake_pred, fake_y, out=np.full_like(fake_y, np.nan), where=(fake_y > 0))

# Is the excess tagger-specific or kinematic? Compare to the all-tagged (>= LP) ratio.
TAGGED = ["HP", "MP", "LP"]
data_tag = sum(data_excl[c][0] for c in TAGGED)
real_tag = sum(real_excl_predicted_nom[c] for c in TAGGED)
fake_tag = sum(fake_excl_nom[c][0] for c in TAGGED)
sf_incl = np.divide(np.clip(data_tag - real_tag, 0, None), fake_tag,
                    out=np.full_like(fake_tag, np.nan), where=fake_tag > 0)

print(f"\nGlobal CR data/MC: {data_tag.sum() / (real_incl_nom['LP'][0].sum() + fake_tag.sum()):.3f}")
print("Inclusive (>=LP) fake data/MC per bin [rows=pT, cols=mSD]:\n", np.round(sf_incl, 3))
for c in TAGGED:
    shape_only = sffake_central[c] / sf_incl
    print(f"{c} shape-only SF (SF_cat / SF_incl):\n", np.round(shape_only, 3))

# --- 1. STATISTICAL: Clopper-Pearson on Data, MC-fake, and MC-real ratios ---
sigma_data, sigma_fake_mc, sigma_scaled_excl = {}, {}, {}
for c in CATEGORIES:
    frac_data = np.divide(data_excl[c][0], data_total_y, out=np.zeros_like(data_total_y), where=(data_total_y > 0))
    lo, hi = clopper_pearson_interval(data_excl[c][0], data_total_y)  # exact counts, no effective-N needed
    sigma_data[c] = 0.5 * (data_total_y * hi - data_total_y * lo)

    frac_fake = np.divide(fake_excl_nom[c][0], fake_total_nom[0], out=np.zeros_like(fake_total_nom[0]), where=(fake_total_nom[0] > 0))
    lo, hi = stat_band_on_ratio(frac_fake, fake_total_nom[0], fake_total_nom[1])
    sigma_fake_mc[c] = 0.5 * (fake_total_nom[0] * hi - fake_total_nom[0] * lo)

    # Real MC: use the DIRECTLY-measured exclusive real yield's own
    # sumw2 for the variance (not the inclusive-difference algebra used
    # for the central value) -- HP/MP/LP-inclusive counts are nested
    # subsets of each other, so naively quadrature-subtracting their
    # individual CP bands would ignore that correlation and overstate
    # the uncertainty. The representative SF for scaling this into
    # "Data-corrected" uncertainty units is approximate: Fail has no
    # associated SF (matches how scaled_incl["Total"] carries no SF
    # factor in the central-value calculation either).
    sf_key_for_unc = {"HP": "HP", "MP": "MP", "LP": "LP", "Fail": None}[c]
    frac_real = np.divide(real_excl_nom[c][0], real_incl_nom["Total"][0], out=np.zeros_like(real_incl_nom["Total"][0]), where=(real_incl_nom["Total"][0] > 0))
    lo, hi = stat_band_on_ratio(frac_real, real_incl_nom["Total"][0], real_incl_nom["Total"][1])
    sigma_R_excl = 0.5 * (real_incl_nom["Total"][0] * hi - real_incl_nom["Total"][0] * lo)
    sf_val = real_sf["nominal"][sf_key_for_unc] if sf_key_for_unc else 1.0
    sigma_scaled_excl[c] = sf_val * sigma_R_excl

stat_unc = {}
for c in CATEGORIES:
    fake_y = fake_excl_nom[c][0]
    sigma_num = np.sqrt(sigma_data[c]**2 + sigma_scaled_excl[c]**2)
    stat_unc[c] = np.divide(
        np.sqrt(sigma_num**2 + (sffake_central[c] * sigma_fake_mc[c])**2),
        fake_y, out=np.full_like(fake_y, np.nan), where=(fake_y > 0)
    )

# --- 2. SYSTEMATIC: +-20% signal (ttZ, ttH) rate ---
# Applied to BOTH the real-subtraction term and the fake-MC denominator
# -- a ttZ/ttH event can land in either population depending on whether
# its candidate GEN-matches, so "varying the signal rate" has to touch both.
real_incl_sigUp   = {w: combine_processes(real_incl_by_proc[w], sig_processes, 1 + SIGNAL_RATE_VARIATION) for w in REAL_THRESH_KEYS}
real_incl_sigDown = {w: combine_processes(real_incl_by_proc[w], sig_processes, 1 - SIGNAL_RATE_VARIATION) for w in REAL_THRESH_KEYS}
fake_excl_sigUp   = {c: combine_processes(fake_excl_by_proc[c], sig_processes, 1 + SIGNAL_RATE_VARIATION) for c in CATEGORIES}
fake_excl_sigDown = {c: combine_processes(fake_excl_by_proc[c], sig_processes, 1 - SIGNAL_RATE_VARIATION) for c in CATEGORIES}

real_excl_pred_sigUp   = build_exclusive_real_prediction(real_incl_sigUp, real_sf["nominal"])
real_excl_pred_sigDown = build_exclusive_real_prediction(real_incl_sigDown, real_sf["nominal"])

sigRate_up, sigRate_down = {}, {}
for c in CATEGORIES:
    fy_up, fy_dn = fake_excl_sigUp[c][0], fake_excl_sigDown[c][0]
    df_up = np.clip(data_excl[c][0] - real_excl_pred_sigUp[c], 0.0, None)
    df_dn = np.clip(data_excl[c][0] - real_excl_pred_sigDown[c], 0.0, None)
    sf_up = np.divide(df_up, fy_up, out=np.full_like(fy_up, np.nan), where=(fy_up > 0))
    sf_dn = np.divide(df_dn, fy_dn, out=np.full_like(fy_dn, np.nan), where=(fy_dn > 0))
    sigRate_up[c] = np.abs(sf_up - sffake_central[c])
    sigRate_down[c] = np.abs(sffake_central[c] - sf_dn)

# --- 3. SYSTEMATIC: real-SF uncertainty propagated through ---
real_excl_pred_realUp   = build_exclusive_real_prediction(real_incl_nom, real_sf["up"])
real_excl_pred_realDown = build_exclusive_real_prediction(real_incl_nom, real_sf["down"])

realSF_up, realSF_down = {}, {}
for c in CATEGORIES:
    fake_y = fake_excl_nom[c][0]
    df_up = np.clip(data_excl[c][0] - real_excl_pred_realUp[c], 0.0, None)
    df_dn = np.clip(data_excl[c][0] - real_excl_pred_realDown[c], 0.0, None)
    sf_up = np.divide(df_up, fake_y, out=np.full_like(fake_y, np.nan), where=(fake_y > 0))
    sf_dn = np.divide(df_dn, fake_y, out=np.full_like(fake_y, np.nan), where=(fake_y > 0))
    realSF_up[c] = np.abs(sf_up - sffake_central[c])
    realSF_down[c] = np.abs(sffake_central[c] - sf_dn)

# --- Quadrature total ---
total_up, total_down = {}, {}
for c in CATEGORIES:
    total_up[c] = np.sqrt(stat_unc[c]**2 + sigRate_up[c]**2 + realSF_up[c]**2)
    total_down[c] = np.sqrt(stat_unc[c]**2 + sigRate_down[c]**2 + realSF_down[c]**2)

results = {
    'pt_bins': PT_BINS, 'msd_bins': MSD_BINS,
    'central': sffake_central, 'stat_unc': stat_unc,
    'sigRate_up': sigRate_up, 'sigRate_down': sigRate_down,
    'realSF_up': realSF_up, 'realSF_down': realSF_down,
    'total_up': total_up, 'total_down': total_down,
}

# --- Diagnostics: is statistical actually dominant, as claimed? ---
print(f"\n--- Derived Exclusive Fake SFs ({TARGET_ERA}) ---")
for c in CATEGORIES:
    valid = ~np.isnan(sffake_central[c])
    n_valid = int(valid.sum())
    if c == "Fail" and n_valid == 0:
        print(f"\n{c}: 0 valid bins -- EXPECTED, not a bug. The baseline selection "
              f"requires ZH_bbvLscore >= LP, so no candidate can ever fall below "
              f"Loose in this CR; see cuts() docstring.")
        continue
    print(f"\n{c}, mean over {n_valid} valid bins:")
    vals = sffake_central[c][valid]
    if vals.size:
        print(f"  SFfake central        = mean {np.mean(vals):.3f} | median {np.median(vals):.3f} "
              f"| min {np.min(vals):.3f} | max {np.max(vals):.3f}")
        print(f"                          (median is far more robust than mean to a single "
              f"low-stat outlier bin -- large mean/median gap = check the flagged bins below)")

    # A near-zero MC-fake denominator in even one bin can dominate the
    # mean above. Flag bins with few effective MC events directly,
    # cross-referenced against that same bin's stat_unc -- if stat_unc
    # is similarly huge there, the large central value is a genuine
    # low-statistics artifact (correctly flagged as unreliable by its
    # own uncertainty), not a hidden bug elsewhere.
    fake_yield, fake_sumw2 = fake_excl_nom[c]
    n_eff_fake = np.divide(fake_yield**2, fake_sumw2, out=np.zeros_like(fake_yield), where=(fake_sumw2 > 0))
    low_stat_mask = valid & (n_eff_fake < 10)
    if np.any(low_stat_mask):
        idx = np.argwhere(low_stat_mask)
        print(f"  \u26A0\uFE0F  {len(idx)} bin(s) with < 10 effective MC fake events -- likely unreliable:")
        for j, k in idx:
            print(f"      pT[{PT_BINS[j]:.0f},{PT_BINS[j+1]:.0f}) mSD[{MSD_BINS[k]:.0f},{MSD_BINS[k+1]:.0f}) "
                  f"N_eff={n_eff_fake[j, k]:.1f}  SFfake={sffake_central[c][j, k]:.3f}  "
                  f"stat_unc={stat_unc[c][j, k]:.3f}")

    print(f"  stat unc (CP)         = {np.nanmean(stat_unc[c]):.4f}")
    print(f"  sigRate unc (+/-20%)  = {np.nanmean(sigRate_up[c]):.4f} / {np.nanmean(sigRate_down[c]):.4f}")
    print(f"  realSF-propagated unc = {np.nanmean(realSF_up[c]):.4f} / {np.nanmean(realSF_down[c]):.4f}")
    print(f"  total unc             = {np.nanmean(total_up[c]):.4f} / {np.nanmean(total_down[c]):.4f}")
    if np.nanmean(stat_unc[c]) < max(np.nanmean(sigRate_up[c]), np.nanmean(realSF_up[c])):
        print(f"  \u26A0\uFE0F  Statistical is NOT the largest source for {c} -- check against the methodology text's claim.")


def check_msd_flatness(sffake_central, stat_unc, categories, pt_bins, msd_bins):
    """
    For each pT bin, tests whether SFfake is statistically consistent
    with being flat across mSD bins: an inverse-variance-weighted
    chi-square against stat_unc alone. NOT tested against the
    systematics -- realSF_up/down is fully correlated across all mSD
    bins at fixed pT (SF_real doesn't depend on mSD, so it shifts every
    mSD bin together and can't explain genuine bin-to-bin scatter), and
    sigRate represents a model variation, not measurement noise. Only
    stat_unc answers "how much would each mSD bin wobble independently
    if this measurement were repeated" -- which is the actual question.

    chi2/dof well above 1 (small p-value): the mSD dependence looks
    like a real trend, not noise -- evidence AGAINST collapsing the
    mSD binning for this category/pT bin.
    chi2/dof near 1: the scatter across mSD bins is consistent with
    pure statistical noise -- it would be safe to coarsen or drop the
    mSD axis there.
    """
    print("\n--- mSD flatness check (per pT bin, stat-only) ---")
    for c in categories:
        central, unc = sffake_central[c], stat_unc[c]
        print(f"\n{c}:")
        for j in range(len(pt_bins) - 1):
            row_central, row_unc = central[j, :], unc[j, :]
            valid = ~np.isnan(row_central) & ~np.isnan(row_unc) & (row_unc > 0)
            n_valid = int(valid.sum())
            pt_label = f"pT[{pt_bins[j]:.0f},{pt_bins[j+1]:.0f})"
            if n_valid < 2:
                print(f"  {pt_label}: only {n_valid} valid mSD bin(s) -- can't test flatness")
                continue
            vals, sig = row_central[valid], row_unc[valid]
            weights = 1.0 / sig**2
            weighted_mean = np.sum(weights * vals) / np.sum(weights)
            chi2_stat = np.sum(weights * (vals - weighted_mean)**2)
            dof = n_valid - 1
            p_value = chi2_dist.sf(chi2_stat, dof) if dof > 0 else np.nan
            flag = "  \u26A0\uFE0F LIKELY REAL mSD TREND" if (dof > 0 and p_value < 0.05) else ""
            print(f"  {pt_label}: chi2/dof = {chi2_stat/dof:.2f} (dof={dof}), p={p_value:.3f}, "
                  f"weighted mean SFfake = {weighted_mean:.3f}{flag}")
            if flag:
                # Show exactly which mSD bin(s) are driving the trend, and by
                # how many sigma -- distinguishes a robust, significant bin
                # (large pull, tight uncertainty) from one that just LOOKS
                # dramatic but is statistically consistent with the mean.
                for k in range(len(msd_bins) - 1):
                    if not valid[k]:
                        print(f"      mSD[{msd_bins[k]:.0f},{msd_bins[k+1]:.0f}): n/a (undefined bin)")
                        continue
                    pull = (row_central[k] - weighted_mean) / row_unc[k]
                    print(f"      mSD[{msd_bins[k]:.0f},{msd_bins[k+1]:.0f}): "
                          f"SFfake = {row_central[k]:.3f} +/- {row_unc[k]:.3f}  "
                          f"(pull = {pull:+.2f} sigma from weighted mean)")


check_msd_flatness(sffake_central, stat_unc, CATEGORIES, PT_BINS, MSD_BINS)

# ==========================================
# CELL 5: SAVE TO CORRECTIONLIB
# ==========================================
def save_to_correctionlib(results, output_file, era_label):
    pt_edges = [float(x) for x in results['pt_bins']]
    msd_edges = [float(x) for x in results['msd_bins']]

    def grid_for(central, delta=None, sign=1):
        g = central if delta is None else central + sign * delta
        return np.nan_to_num(g, nan=1.0)  # correctionlib can't store 'undefined'

    wp_contents = []
    for cat in CATEGORIES:
        c = results['central'][cat]
        syst_grids = {
            'nominal':     grid_for(c),
            'statUp':      grid_for(c, results['stat_unc'][cat], +1),
            'statDown':    grid_for(c, results['stat_unc'][cat], -1),
            'sigRateUp':   grid_for(c, results['sigRate_up'][cat], +1),
            'sigRateDown': grid_for(c, results['sigRate_down'][cat], -1),
            'realSFUp':    grid_for(c, results['realSF_up'][cat], +1),
            'realSFDown':  grid_for(c, results['realSF_down'][cat], -1),
            'up':          grid_for(c, results['total_up'][cat], +1),
            'down':        grid_for(c, results['total_down'][cat], -1),
        }
        syst_contents = []
        for syst_key, grid in syst_grids.items():
            msd_binning_per_pt = [
                cs.Binning(nodetype="binning", input="fatjet_msd", edges=msd_edges,
                           content=[float(v) for v in grid[j, :]], flow="clamp")
                for j in range(len(pt_edges) - 1)
            ]
            pt_binning = cs.Binning(nodetype="binning", input="jet_pt", edges=pt_edges,
                                     content=msd_binning_per_pt, flow="clamp")
            syst_contents.append({"key": syst_key, "value": pt_binning})
        syst_category = cs.Category(nodetype="category", input="systematic", content=syst_contents)
        wp_contents.append({"key": cat, "value": syst_category})

    correction = cs.Correction(
        name=f"Fake_SF_Xbb_{era_label}",
        version=1,
        description=(
            f"Exclusive fake Xbb scale factors for {era_label}. WorkingPoint is the "
            "exclusive tagger category (HP/MP/LP/Fail). systematic keys: 'nominal', "
            "'statUp'/'statDown' (Clopper-Pearson), 'sigRateUp'/'sigRateDown' "
            "(+/-20% ttZ+ttH rate), 'realSFUp'/'realSFDown' (real-SF uncertainty "
            "propagated through), 'up'/'down' (quadrature total)."
        ),
        inputs=[
            cs.Variable(name="WorkingPoint", type="string", description="Exclusive category: 'HP', 'MP', 'LP', or 'Fail'"),
            cs.Variable(name="systematic", type="string", description="See Correction description for the full list of keys"),
            cs.Variable(name="jet_pt", type="real", description="Z/H candidate transverse momentum [GeV]"),
            cs.Variable(name="fatjet_msd", type="real", description="Z/H candidate soft-drop mass [GeV]"),
        ],
        output=cs.Variable(name="weight", type="real", description="Multiplicative fake-candidate scale factor"),
        data=cs.Category(nodetype="category", input="WorkingPoint", content=wp_contents),
    )
    cset = cs.CorrectionSet(schema_version=2, corrections=[correction])
    with open(output_file, "w") as f:
        f.write(cset.model_dump_json(exclude_unset=True, indent=2))
    with gzip.open(f"{output_file}.gz", "wt") as f:
        f.write(cset.model_dump_json(exclude_unset=True))
    print(f"\nGenerated {output_file} and {output_file}.gz")


save_to_correctionlib(results, OUTPUT_FAKE_SF_FILE, TARGET_ERA)
print(f"\n--- Complete ---")

✅ Configuration loaded.

--- Starting Data Extraction ---
Extracting from 5 nominal files...
Data events passing CR baseline:     12,608
MC Real events passing CR baseline:  155,188  (processes: ['VJets', 'tt_B', 'TTBar', 'SingleTop', 'TTX', 'ttZ', 'ttH'])
MC Fake events passing CR baseline:  517,429  (processes: ['VJets', 'tt_B', 'TTBar', 'SingleTop', 'TTX', 'ttZ', 'ttH'])

Global CR data/MC: 1.166
Inclusive (>=LP) fake data/MC per bin [rows=pT, cols=mSD]:
 [[1.36  1.152 1.142 1.249]
 [1.257 1.186 1.15  1.04 ]
 [1.291 1.081 1.162 1.066]]
HP shape-only SF (SF_cat / SF_incl):
 [[1.011 1.077 0.904 1.016]
 [0.988 1.05  1.101 0.922]
 [1.823 0.977 1.206 1.055]]
MP shape-only SF (SF_cat / SF_incl):
 [[1.071 0.944 0.953 0.809]
 [1.2   0.867 0.932 1.05 ]
 [0.713 0.905 0.924 1.139]]
LP shape-only SF (SF_cat / SF_incl):
 [[0.983 1.005 1.029 1.073]
 [0.955 1.029 1.006 0.996]
 [0.928 1.029 0.993 0.949]]

--- Derived Exclusive Fake SFs (2017) ---

HP, mean over 12 valid bins:
  SFfake central      